# Customer Support Intelligence Platform — Phase 5: BiLSTM Deep Learning Model

**Brief requirements covered (Day 11):**
- Build a PyTorch BiLSTM model using **pre-trained GloVe embeddings (100d)**
  on Ticket Description text
- Train for Ticket Type multi-class classification
- Apply BatchNorm and Dropout
- Apply early stopping
- Compare against classical ML baseline

**Using GloVe here specifically (not Word2Vec):** Day 5-6 used self-trained
Word2Vec as the brief's allowed alternative for that step. This step
specifically calls for GloVe, so we use genuine pretrained embeddings this
time - downloaded via gensim's downloader API (glove-wiki-gigaword-100),
which fetches the same Stanford GloVe vectors without needing a manual
822MB zip download.

**Expectation, set honestly from prior results:** every classical model
(6 total, across 2 notebooks) converged to near-baseline performance,
confirming a weak-signal dataset. A BiLSTM might extract marginally
different signal through sequential processing, but a dramatic jump would
be surprising - we report whatever we actually find.

In [1]:

import sys
sys.path.insert(0, "../src")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import gensim.downloader as gensim_api
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import mlflow
import mlflow.pytorch

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_df = pd.read_csv("../data_processed/train.csv")
val_df = pd.read_csv("../data_processed/val.csv")
test_df = pd.read_csv("../data_processed/test.csv")
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

Using device: cuda
Train: 5928, Val: 1270, Test: 1271


## 1. Download pretrained GloVe 100d embeddings

**Why this specific source:** `glove-wiki-gigaword-100` via gensim's
downloader is the same Stanford GloVe vectors (trained on Wikipedia +
Gigaword), pre-converted to a format gensim can load directly - avoids
manually downloading and parsing Stanford's raw 822MB multi-dimension zip
file. First run downloads (~128MB, cached afterward).

In [2]:
print("Downloading GloVe 100d embeddings (cached after first run)...")
glove_vectors = gensim_api.load("glove-wiki-gigaword-100")
print(f"GloVe vocabulary size: {len(glove_vectors)}")
print(f"Embedding dimension: {glove_vectors.vector_size}")

GloVe vocabulary size: 400000
Embedding dimension: 100


## 2. Build vocabulary and embedding matrix

Vocabulary built from our OWN training data's cleaned text - only words that
actually appear in our tickets get a slot. For each vocabulary word, we look
up its GloVe vector if available; words GloVe doesn't know (rare/misspelled
words) get a random initialization instead of failing.

In [3]:
MAX_VOCAB_SIZE = 8000
MAX_SEQ_LENGTH = 100  # truncate/pad descriptions to this many tokens
EMBEDDING_DIM = 100

all_train_tokens = []
for text in train_df["Description_Clean"].fillna(""):
    all_train_tokens.extend(text.split())

word_counts = Counter(all_train_tokens)
most_common = word_counts.most_common(MAX_VOCAB_SIZE - 2)  # reserve 2 slots for PAD/UNK

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for word, _ in most_common:
    word_to_idx[word] = len(word_to_idx)

vocab_size = len(word_to_idx)
print(f"Vocabulary size (from training data): {vocab_size}")

embedding_matrix = np.random.normal(0, 0.1, (vocab_size, EMBEDDING_DIM)).astype(np.float32)
embedding_matrix[0] = np.zeros(EMBEDDING_DIM)  # PAD token = all zeros

found_in_glove = 0
for word, idx in word_to_idx.items():
    if word in glove_vectors:
        embedding_matrix[idx] = glove_vectors[word]
        found_in_glove += 1

print(f"Words found in GloVe: {found_in_glove}/{vocab_size} ({100*found_in_glove/vocab_size:.1f}%)")

Vocabulary size (from training data): 4883
Words found in GloVe: 3776/4883 (77.3%)


## 3. PyTorch Dataset and text-to-sequence conversion

In [4]:
def text_to_sequence(text, word_to_idx, max_len):
    tokens = text.split() if isinstance(text, str) else []
    seq = [word_to_idx.get(tok, word_to_idx["<UNK>"]) for tok in tokens[:max_len]]
    seq = seq + [word_to_idx["<PAD>"]] * (max_len - len(seq))
    return seq

type_label_map = {label: i for i, label in enumerate(sorted(train_df["Ticket Type"].unique()))}
NUM_CLASSES = len(type_label_map)

class TicketDataset(Dataset):
    def __init__(self, df, word_to_idx, max_len, label_map):
        self.sequences = [text_to_sequence(t, word_to_idx, max_len) for t in df["Description_Clean"]]
        self.labels = [label_map[l] for l in df["Ticket Type"]]

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)

train_dataset = TicketDataset(train_df, word_to_idx, MAX_SEQ_LENGTH, type_label_map)
val_dataset = TicketDataset(val_df, word_to_idx, MAX_SEQ_LENGTH, type_label_map)
test_dataset = TicketDataset(test_df, word_to_idx, MAX_SEQ_LENGTH, type_label_map)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Train batches: 186, Val batches: 40


## 4. BiLSTM model architecture

**Matches brief spec exactly:** GloVe embedding layer (initialized with our
matrix, fine-tuned during training) -> Bidirectional LSTM -> BatchNorm ->
Dropout -> Linear classification head.

In [5]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, embedding_matrix, hidden_dim=64, num_classes=5, dropout=0.5):
        super().__init__()
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=False, padding_idx=0
        )
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.batch_norm = nn.BatchNorm1d(hidden_dim * 2)  # *2 for bidirectional
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, (hidden, _) = self.lstm(embedded)
        # Concatenate final forward and backward hidden states
        final_hidden = torch.cat([hidden[0], hidden[1]], dim=1)
        normalized = self.batch_norm(final_hidden)
        dropped = self.dropout(normalized)
        return self.fc(dropped)

model = BiLSTMClassifier(embedding_matrix, hidden_dim=64, num_classes=NUM_CLASSES, dropout=0.5).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(model)

Total parameters: 574,193
BiLSTMClassifier(
  (embedding): Embedding(4883, 100, padding_idx=0)
  (lstm): LSTM(100, 64, batch_first=True, bidirectional=True)
  (batch_norm): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=128, out_features=5, bias=True)
)


## 5. Training loop with early stopping

Early stopping: halts training if validation loss doesn't improve for
5 consecutive epochs, keeping the best checkpoint.

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

MAX_EPOCHS = 30
PATIENCE = 5

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.set_grad_enabled(is_train):
        for sequences, labels in loader:
            sequences, labels = sequences.to(device), labels.to(device)
            if is_train:
                optimizer.zero_grad()
            outputs = model(sequences)
            loss = criterion(outputs, labels)
            if is_train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * sequences.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

import copy
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss = float("inf")
best_model_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

for epoch in range(MAX_EPOCHS):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion, optimizer=None)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
        marker = " <- best so far"
    else:
        epochs_without_improvement += 1
        marker = f" ({epochs_without_improvement}/{PATIENCE} epochs without improvement)"

    print(f"Epoch {epoch+1:2d}/{MAX_EPOCHS} | train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
          f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}{marker}")

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}.")
        break

model.load_state_dict(best_model_state)
print(f"\nLoaded best model (val_loss={best_val_loss:.4f})")

Epoch  1/30 | train_loss=1.6750 train_acc=0.2028 | val_loss=1.6773 val_acc=0.2031 <- best so far
Epoch  2/30 | train_loss=1.6369 train_acc=0.2051 | val_loss=1.7126 val_acc=0.1929 (1/5 epochs without improvement)
Epoch  3/30 | train_loss=1.6246 train_acc=0.2093 | val_loss=1.7644 val_acc=0.1937 (2/5 epochs without improvement)
Epoch  4/30 | train_loss=1.6107 train_acc=0.2230 | val_loss=1.6710 val_acc=0.1921 <- best so far
Epoch  5/30 | train_loss=1.5907 train_acc=0.2566 | val_loss=1.7999 val_acc=0.1866 (1/5 epochs without improvement)
Epoch  6/30 | train_loss=1.5442 train_acc=0.2944 | val_loss=1.7913 val_acc=0.2024 (2/5 epochs without improvement)
Epoch  7/30 | train_loss=1.4712 train_acc=0.3387 | val_loss=2.0579 val_acc=0.2071 (3/5 epochs without improvement)
Epoch  8/30 | train_loss=1.3655 train_acc=0.3990 | val_loss=2.0954 val_acc=0.1984 (4/5 epochs without improvement)
Epoch  9/30 | train_loss=1.2528 train_acc=0.4514 | val_loss=2.1048 val_acc=0.1921 (5/5 epochs without improvement)



## 6. Evaluation on test set

In [7]:
type_label_map_inv = {v: k for k, v in type_label_map.items()}

model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for sequences, labels in test_loader:
        sequences = sequences.to(device)
        outputs = model(sequences)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

test_acc = accuracy_score(all_labels, all_preds)
test_f1_macro = f1_score(all_labels, all_preds, average="macro")
try:
    test_roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr")
except ValueError:
    test_roc_auc = None

print(f"BiLSTM Test Accuracy: {test_acc:.4f}")
print(f"BiLSTM Test F1-macro: {test_f1_macro:.4f}")
print(f"BiLSTM Test ROC-AUC:  {test_roc_auc:.4f}" if test_roc_auc else "ROC-AUC: N/A")
print()
label_names = [type_label_map_inv[i] for i in range(NUM_CLASSES)]
print(classification_report(all_labels, all_preds, target_names=label_names))

BiLSTM Test Accuracy: 0.1943
BiLSTM Test F1-macro: 0.0691
BiLSTM Test ROC-AUC:  0.4871

                      precision    recall  f1-score   support

     Billing inquiry       0.19      1.00      0.32       245
Cancellation request       0.00      0.00      0.00       255
     Product inquiry       0.00      0.00      0.00       246
      Refund request       0.00      0.00      0.00       263
     Technical issue       0.43      0.01      0.02       262

            accuracy                           0.19      1271
           macro avg       0.12      0.20      0.07      1271
        weighted avg       0.13      0.19      0.07      1271



c:\Users\CHARU\OneDrive\Desktop\Final Project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\CHARU\OneDrive\Desktop\Final Project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\CHARU\OneDrive\Desktop\Final Project\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{m

## 7. Comparison against all prior models

**Note:** the classical-model accuracy figures below are copied in from the
results recorded in `04_Baseline_Models.ipynb` / `06_Ensemble_Models.ipynb`.
Verify they still match those notebooks' current output before submission
(or better, replace this block with an MLflow query against the
`customer_support_classical_ml` experiment so the numbers can't go stale).
This cell reports the numbers plainly rather than asserting whether any
gap counts as a "meaningful" improvement, since we have no significance
test to back that judgment.

In [8]:
print("FULL COMPARISON: Ticket Type Classification Across All Models")
print("=" * 65)
print(f"{'Model':<40s} {'Accuracy':>10s} {'F1-macro':>10s}")
print("-" * 61)
# TODO: confirm these against 04_Baseline_Models.ipynb / 06_Ensemble_Models.ipynb
# before final submission, or pull them from MLflow directly.
print(f"{'Logistic Regression (baseline)':<40s} {0.2071:>10.4f} {'n/a':>10s}")
print(f"{'Naive Bayes (baseline)':<40s} {0.2134:>10.4f} {'n/a':>10s}")
print(f"{'Random Forest (tabular+TFIDF)':<40s} {0.1850:>10.4f} {'n/a':>10s}")
print(f"{'XGBoost (Optuna-tuned)':<40s} {0.2102:>10.4f} {'n/a':>10s}")
print(f"{'BiLSTM (GloVe embeddings)':<40s} {test_acc:>10.4f} {test_f1_macro:>10.4f}")
print()
print(f"Random guessing baseline for {NUM_CLASSES} classes: {1/NUM_CLASSES:.4f}")
print()
print("All figures reported as-is; see the final report's Limitations section")
print("for the shuffle-test evidence establishing this dataset's weak learnable")
print("signal for Ticket Type across every model architecture tried so far.")

FULL COMPARISON: Ticket Type Classification Across All Models
Model                                      Accuracy   F1-macro
-------------------------------------------------------------
Logistic Regression (baseline)               0.2071        n/a
Naive Bayes (baseline)                       0.2134        n/a
Random Forest (tabular+TFIDF)                0.1850        n/a
XGBoost (Optuna-tuned)                       0.2102        n/a
BiLSTM (GloVe embeddings)                    0.1943     0.0691

Random guessing baseline for 5 classes: 0.2000

All figures reported as-is; see the final report's Limitations section
for the shuffle-test evidence establishing this dataset's weak learnable
signal for Ticket Type across every model architecture tried so far.


## 8. Save model and log to MLflow

In [9]:
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("customer_support_deep_learning")

with mlflow.start_run(run_name="bilstm_glove_ticket_type"):
    mlflow.log_param("model_type", "BiLSTM")
    mlflow.log_param("embedding", "GloVe-100d (glove-wiki-gigaword-100)")
    mlflow.log_param("hidden_dim", 64)
    mlflow.log_param("dropout", 0.5)
    mlflow.log_param("max_seq_length", MAX_SEQ_LENGTH)
    mlflow.log_param("vocab_size", vocab_size)
    mlflow.log_param("epochs_trained", len(history["train_loss"]))
    mlflow.log_param("early_stopping_patience", PATIENCE)
    mlflow.log_param("random_state", RANDOM_STATE)
    mlflow.log_metric("test_accuracy", test_acc)
    mlflow.log_metric("test_f1_macro", test_f1_macro)
    if test_roc_auc:
        mlflow.log_metric("test_roc_auc", test_roc_auc)
    mlflow.pytorch.log_model(model, "model")

torch.save(model.state_dict(), "../models/bilstm_glove_model.pth")
import json
with open("../models/bilstm_vocab.json", "w") as f:
    json.dump(word_to_idx, f)

print("Saved model to models/bilstm_glove_model.pth")
print("Saved vocabulary to models/bilstm_vocab.json")
print("Logged run to MLflow experiment 'customer_support_deep_learning'")

2026/09/11 16:05:55 INFO mlflow.tracking.fluent: Experiment with name 'customer_support_deep_learning' does not exist. Creating a new experiment.
2026/09/11 16:05:55 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/09/11 16:06:08 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\CHARU\AppData\Local\Temp\tmp1vr3e22l\model\data, flavor: pytorch). Fall back to return ['torch==2.11.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 
2026/09/11 16:06:08 WARNING mlflow.models.model: Input example should be provided to infer model signature if the model signature is not provided when lo

Saved model to models/bilstm_glove_model.pth
Saved vocabulary to models/bilstm_vocab.json
Logged run to MLflow experiment 'customer_support_deep_learning'
